# Interrupt Loop Debug

Ask -> LLM -> Ask cycle with a mock Slack server.  
Uses checkpointer-based pause/resume with `workflow_id` for proper state persistence across turns.

In [ ]:
import threading
from typing import Any

from examples.slack_mock import MockSlackServer, SlackClient
from hypergraph import END, AsyncRunner, Graph, PauseInfo, interrupt, node, route
from hypergraph.checkpointers import SqliteCheckpointer

## Helpers

In [ ]:
def fake_llm(messages: list[str]) -> str:
    """Placeholder LLM: echoes the latest user message."""
    latest = next((m for m in reversed(messages) if m.startswith("user: ")), "user: <none>")
    return f"assistant draft based on [{latest}]"


async def run_interactive(
    runner: AsyncRunner,
    graph: Graph,
    receive: Any,
    *,
    workflow_id: str = "slack-chat",
    max_pauses: int = 30,
) -> Any:
    """Run a graph with checkpointer-based pause/resume.

    Each pause persists state via workflow_id. Resume passes only the
    interrupt response — the checkpointer restores everything else.
    """
    # First run: starts the workflow
    result = await runner.run(graph, workflow_id=workflow_id)

    for _ in range(max_pauses):
        if not result.paused:
            return result

        pause = result.pause
        print(f"[demo] paused at '{pause.node_name}', waiting for reply...")
        response = await receive(pause)
        print(f"[demo] received -> {pause.response_key}={response!r}")

        # Resume: only pass the response. Checkpointer restores the rest.
        result = await runner.run(
            graph,
            {pause.response_key: response},
            workflow_id=workflow_id,
        )

    raise RuntimeError(f"Exceeded max_pauses={max_pauses}.")

## Node definitions

All functions are portable: `slack` and `question` are injected via `values`, not captured from globals.

## Graph construction

In [ ]:
@interrupt(output_name="user_input")
def ask_slack(messages: list[str], slack) -> None:
    turn_number = sum(1 for m in messages if m.startswith("assistant: ")) + 1
    prompt = f"Turn {turn_number}. Prior context:\n" + "\n".join(messages[-4:])
    slack.post_message(prompt)
    return None


@node(output_name="assistant_text")
def llm_step(messages: list[str]) -> str:
    return fake_llm(messages)


@node(output_name="messages")
def add_user_message(messages: list[str], user_input: str) -> list[str]:
    return [*messages, f"user: {user_input}"]


@node(output_name="messages")
def add_assistant_message(messages: list[str], assistant_text: str) -> list[str]:
    return [*messages, f"assistant: {assistant_text}"]


@route(targets=["ask_slack", END])
def should_continue(messages: list[str], max_turns: int) -> str:
    turns = sum(1 for m in messages if m.startswith("assistant: "))
    return END if turns >= max_turns else "ask_slack"

In [ ]:
graph = Graph(
    [ask_slack, add_user_message, llm_step, add_assistant_message, should_continue],
    edges=[(add_user_message, llm_step), (add_assistant_message, should_continue)],
    name="slack_cycle",
    shared=["messages"],
    entrypoint="ask_slack",
)

In [ ]:
graph.visualize()

## Run

Start mock Slack server in a background thread, then execute.

**In another terminal**, run:
```
uv run python examples/slack_chat.py
```
to watch messages and type replies interactively.

In [ ]:
# Config
SLACK_URL = "http://127.0.0.1:8765"
max_turns = 3

# Start mock server
server = MockSlackServer(port=8765)
threading.Thread(target=server.serve_forever, daemon=True).start()

# Create client
slack = SlackClient(SLACK_URL)

In [ ]:
async def receive_from_slack(_: PauseInfo) -> str:
    return await slack.receive_response()


# Pre-queue replies so the demo runs without manual input
for i in range(max_turns):
    slack.queue_response(f"User reply {i + 1}")

In [ ]:
graph = graph.bind(messages=[], max_turns=3, slack=slack)

checkpointer = SqliteCheckpointer(":memory:")
await checkpointer.initialize()

In [ ]:
runner = AsyncRunner(checkpointer=checkpointer)
result = await run_interactive(runner, graph, receive_from_slack)

In [ ]:
result

In [ ]:
print("[final messages]")
for line in result["messages"]:
    print(f"  {line}")

print("\n[last run log]")
result.log

## Run 2 — Different config

Same graph, fewer turns and different replies. Both runs share the same checkpointer.

In [ ]:
# Second run: 2 turns with different replies
graph2 = graph.bind(messages=["user: Hi there!"], max_turns=2, slack=slack)

for i in range(2):
    slack.queue_response(f"Short reply {i + 1}")

result2 = await run_interactive(runner, graph2, receive_from_slack, workflow_id="quick-chat")

In [ ]:
result2

## Workflow History

Just type `checkpointer` — expand any run to see the full step-by-step history across all pause/resume cycles, including paused steps.

In [ ]:
checkpointer

In [ ]:
server.close()
await checkpointer.close()